# Machine Learning-Based Prediction of Polycystic Ovary Syndrome (PCOS)

**Group 5** — Fatema Ferdous (0152410052), Wafa Haque (0152420023), Khondoker Sazzad Sunfi (0152310002)

---

This notebook walks through the full study. Each section maps onto one of the objectives
in the project proposal:

| Section | What it covers |
|---|---|
| 1–2 | Preprocessing and EDA pipeline |
| 3–4 | Train/test split, then feature selection: correlation, Chi-Square, RFE |
| 5 | Nine models under repeated stratified k-fold CV |
| 5b–5c | Paired significance tests, and probability calibration |
| 6–7 | Class-imbalance ablation, held-out test evaluation |
| 8 | **Leakage experiment** — measuring how much a flawed protocol inflates accuracy |
| 9 | Explainable AI with SHAP |
| 9b | AutoML benchmark: is there headroom left? |
| 9c | **Cost-tiered screening** — what does each tier of testing actually buy? |
| 9d | Clinical utility (decision curves), uncertainty (bootstrap CIs), subgroups |
| 9e | Learning curves and the 30-split stability sweep |
| 9f | Nested CV — applying our own anti-leakage standard to ourselves |
| 9g | **External validation** on an independent Tunisian cohort |
| 10–11 | Honest comparison against the five reviewed papers; conclusions |

The single most important idea in the notebook is that *every* fitted step —
imputation, scaling, feature selection, and SMOTE — lives inside a pipeline, so it is
refitted on each training fold and never observes the data it is scored on.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append("..")   # so `src` is importable when running from notebooks/

from src import (automl, calibration, clinical, config, data, eda, evaluate, explain,
                 external, features, literature, models, plots, validation)

config.ensure_dirs()
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print("Random seed:", config.RANDOM_STATE)


## 1. Loading and cleaning the data

The dataset is the Kaggle PCOS dataset (Prasoon Kottarathil): 541 patients from 10
hospitals in Kerala, India, with 43 clinical, hormonal and physical measurements.

The raw workbook has several defects that must be fixed before anything will run:

* **Two columns are stored as text** — `II beta-HCG` contains the string `"1.99."`
  (a stray trailing dot) and `AMH` contains a literal `"a"`. Pandas therefore reads
  both entire columns as `object` dtype.
* **`Unnamed: 44`** is an empty column with 2 stray values out of 541.
* **Column names carry inconsistent whitespace** (`" Age (yrs)"`, `"Height(Cm) "`).
* **`Cycle(R/I)`** is coded 2 = regular, 4 = irregular, but one row contains a 5,
  which is not a defined code.
* **Impossible zeros** in `Cycle length(days)` and `Endometrium (mm)` — a 0 mm
  endometrium is a recording failure, not a measurement.


In [ ]:
raw = data.load_raw()
print("Raw shape:", raw.shape)
raw.head()


In [ ]:
df = data.clean(raw)
X, y = data.split_xy(df)

print(f"Cleaned shape : {df.shape}  ({X.shape[1]} features + target)")
print(f"Class balance : {(y==0).sum()} no-PCOS / {(y==1).sum()} PCOS  "
      f"({(y==0).sum()/(y==1).sum():.2f} : 1)")
print(f"Missing values: {df.isna().sum().sum()} across {(df.isna().sum()>0).sum()} columns")

data.data_quality_report(raw, df).head(10)


Note what is **not** happening here: no imputation. The 8 remaining missing values are
left in place deliberately. Filling them now would mean computing a median over the whole
dataset — including the rows we will later use for testing. Imputation belongs inside the
pipeline, fitted per fold.


## 2. Exploratory data analysis

Three questions drive the EDA:

1. How bad is the class imbalance? (It sets up the SMOTE decision.)
2. Which features separate the classes?
3. How much redundancy is there between features? (It motivates feature selection.)


In [ ]:
fig_paths = eda.run_all(df)
for p in fig_paths:
    print(p.name)


In [ ]:
from IPython.display import Image, display
display(Image(str(config.FIGURES_DIR / "01_class_balance.png")))
display(Image(str(config.FIGURES_DIR / "03_target_correlations.png")))


In [ ]:
stats = eda.summary_statistics(df)
print("Largest standardised differences between the two groups (Cohen's d):")
stats.head(12)


In [ ]:
display(Image(str(config.FIGURES_DIR / "07_follicle_separation.png")))
display(Image(str(config.FIGURES_DIR / "06_symptom_prevalence.png")))


**Reading the EDA.** Follicle counts dominate everything else (Cohen's *d* ≈ 1.5–1.7,
a very large effect). This is expected and reassuring — follicle count is one leg of the
Rotterdam diagnostic criteria, so a model that leans on it is behaving sensibly rather
than exploiting an artefact.

The next tier is the self-reported symptom cluster: skin darkening, hair growth and
weight gain, all around *d* ≈ 1.0. The hormone panels (FSH, LH, TSH, prolactin) are
much weaker individually than their clinical prominence would suggest.

One caveat worth flagging in the write-up: `Cycle length(days)` has a median of 5 and a
range of 2–12. Those are not menstrual *cycle* intervals (which would be ~28 days) —
the column is almost certainly recording the duration of bleeding. The Kaggle
documentation does not clarify this, so we report it as-is and note the ambiguity.


## 3. Train / test split — done first, on purpose

The split happens **before** any feature selection, scaling or resampling decision.
The test set is then untouched until Section 6.


In [ ]:
X_train, X_test, y_train, y_test = evaluate.make_splits(X, y)
print(f"Train: {X_train.shape[0]} rows ({(y_train==1).sum()} PCOS)")
print(f"Test : {X_test.shape[0]} rows ({(y_test==1).sum()} PCOS)")


## 4. Feature selection

Three strategies are compared against an all-features baseline:

* **Correlation** — keep features correlated with the target above a threshold, then drop
  one of every pair of mutually redundant features.
* **Chi-Square** — top-*k* by χ² statistic (min-max scaled first, since χ² needs
  non-negative inputs).
* **RFE** — Recursive Feature Elimination driven by logistic regression.

Each is a scikit-learn transformer, so selection is refitted per fold.


In [ ]:
X_prep = models.build_preprocessor(X_train).fit_transform(X_train, y_train)

for name in ("correlation", "chi2", "rfe"):
    sel = features.build_selector(name, k=15).fit(X_prep, y_train)
    print(f"\n{name.upper()} kept {len(sel.selected_features_)} features:")
    for f in sel.selected_features_:
        print("   ", config.pretty(str(f)))


In [ ]:
feature_comparison = evaluate.compare_feature_sets(
    models.build_all_models, X_train, y_train, k_features=15, repeats=1
)
plots.plot_feature_set_comparison(feature_comparison)
display(Image(str(config.FIGURES_DIR / "09_feature_set_comparison.png")))

summary = feature_comparison.groupby("feature_set")["roc_auc"].agg(["mean", "std", "max"])
best_set = summary["mean"].idxmax()
print(summary.round(4))
print("\nBest feature set:", best_set)


The differences between feature sets are small — a couple of thousandths of AUC, well
inside the fold-to-fold noise. That is itself a result: on this dataset, *which* selection
method you use matters far less than whether you apply it correctly. The reduced sets are
still preferable because 15 features are easier to collect in a clinic than 41, and fewer
parameters means less room to overfit 432 training rows.


## 5. Model comparison under repeated stratified cross-validation

Five models, each wrapped in the full pipeline:

```
impute → scale/encode → select features → SMOTE → classifier
```

Cross-validation is **repeated** (10 folds × 3 repeats). With only ~430 training rows a
single 10-fold run swings by several points depending on the shuffle, so reporting one
run would be reporting noise.


In [ ]:
cv_models = models.build_all_models(X_train, selector=best_set, k_features=15)
cv_results = evaluate.cross_validate_models(cv_models, X_train, y_train)

plots.plot_cv_comparison(cv_results)
display(Image(str(config.FIGURES_DIR / "08_cv_model_comparison.png")))

best_model_name = cv_results.iloc[0]["model"]
print("Best model by CV ROC-AUC:", best_model_name)

cols = ["model", "accuracy", "accuracy_std", "precision", "recall", "f1", "roc_auc", "roc_auc_std"]
cv_results[cols].round(4)


Look at the standard deviations before the means. The models sit within roughly one
standard deviation of each other on ROC-AUC. Rather than assert they are tied, the next
two sections test it.


### 5b. Is the ranking real? Paired significance tests

Every model is scored on **identical folds**, so the per-fold difference isolates the
model rather than the split. Two separate questions get two separate columns:

* `statistically_sep` — did a paired t-test detect *any* consistent difference?
* `practically_sep` — is the gap big enough to matter (>= 0.02 AUC)?

Conflating these is a standard way to over-claim.


In [ ]:
fold_scores = calibration.paired_fold_scores(cv_models, X_train, y_train)
significance = calibration.significance_against_best(fold_scores)
calibration.plot_significance(fold_scores)
display(Image(str(config.FIGURES_DIR / "20_significance_boxplot.png")))
significance.round(4)


**Zero of eight** models differ from the best by a practically meaningful margin. The
largest gap in the table is ~0.012 AUC, against a fold-to-fold standard deviation of
~0.030 — six times larger.

The two columns disagree, and that is the lesson. With 30 folds a paired test flags
XGBoost's 0.009 AUC deficit as "significant" (p = 0.003), but no clinical decision turns
on 0.009 AUC. Statistical separability is not the same as mattering.

Two consequences for the report:

* **The stacking ensemble buys nothing.** Papers 4 and 5 both crown a stacking classifier;
  here it lands 0.001 AUC below a plain Random Forest, while costing far more to train
  and explain.
* **Gaussian Naive Bayes is competitive** — no hyperparameters, a famously wrong
  independence assumption, and still statistically tied with the best. Strong evidence
  that this dataset's signal is simple and additive.

*Caveat:* repeated-CV folds overlap, so scores are not independent and the paired t-test
is optimistic. Large p-values (evidence of no difference) are trustworthy; small ones are
suggestive only. Fortunately that is the direction this argument needs.


### 5c. Calibration — do the predicted probabilities mean anything?

ROC-AUC measures only *ordering*. A model can rank patients perfectly and still claim 0.8
for cases that turn out positive 60% of the time. Every metric in the five reviewed papers
is blind to this, and it is precisely what matters when a probability sets a screening
threshold.

* **Brier score** — a proper scoring rule, minimised only by honest probabilities.
* **ECE** — mean gap between claimed confidence and observed frequency.


In [ ]:
calibration_results = calibration.compare_calibration(cv_models, X_train, y_train, folds=5)
calibration.plot_calibration_curves(cv_models, X_train, y_train, folds=5)
display(Image(str(config.FIGURES_DIR / "19_calibration_curves.png")))
calibration_results.round(4)


This ranking is **not** the ROC-AUC ranking. Gaussian NB is statistically tied with the
best on AUC yet is by far the worst-calibrated — its rankings are fine but its
probabilities are near-useless. XGBoost is second-worst. This is exactly the kind of
difference that should decide between tied models.


In [ ]:
# Does explicit calibration help? Not always.
base = models.classifier_by_name(best_model_name)
effect = calibration.calibration_effect(
    models.build_pipeline(X_train, base, selector=best_set, k_features=15),
    models.build_calibrated(X_train, base, selector=best_set, k_features=15),
    X_train, y_train,
)
effect.round(4)


**A negative result worth reporting.** Isotonic calibration did not help: Brier improves by
a rounding error while log loss rises sharply, ECE gets *worse*, and AUC dips.

Two things are happening. Isotonic regression is non-parametric and needs more than 432
rows to fit a stable mapping, so it overfits the calibration folds. It also pins
probabilities hard at 0 and 1, which log loss punishes severely on the cases it gets
wrong.

Calibration is not free, and on a dataset this size the textbook fix can make the
probabilities less trustworthy rather than more.


## 6. Class imbalance: measure it, don't assume it

None of the five reviewed papers state how they handled the 2:1 imbalance. Rather than
assume SMOTE helps, all three options are compared on the same held-out data.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

def rf(**kwargs):
    return RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                  random_state=config.RANDOM_STATE, n_jobs=-1, **kwargs)

ablation = evaluate.balance_ablation(X_train, y_train, X_test, y_test, rf)
ablation[["strategy", "accuracy", "precision", "recall", "f1", "specificity", "roc_auc"]].round(4)


The effect is real but modest: SMOTE buys a few points of recall — i.e. it catches PCOS
cases the unbalanced model misses — at essentially no cost in specificity. For a screening
tool that trade is worth taking, since a missed diagnosis is worse than a false alarm that
gets resolved by a confirmatory ultrasound.


## 7. Held-out test set

Everything above used only the training split. This is the first and only look at the
test set.


In [ ]:
test_models = models.build_all_models(X_train, selector=best_set, k_features=15)
test_results = evaluate.evaluate_on_test(test_models, X_train, y_train, X_test, y_test)
test_results[["model", "accuracy", "precision", "recall", "f1", "specificity", "roc_auc"]].round(4)


In [ ]:
curves = evaluate.curve_data(test_models, X_test, y_test)
plots.plot_roc_curves(curves)
plots.plot_pr_curves(curves)
display(Image(str(config.FIGURES_DIR / "10_roc_curves.png")))
display(Image(str(config.FIGURES_DIR / "11_pr_curves.png")))


In [ ]:
best_pipeline = test_models[best_model_name]
y_pred = best_pipeline.predict(X_test)

plots.plot_confusion_matrix(y_test, y_pred, best_model_name)
display(Image(str(config.FIGURES_DIR / "12_confusion_matrix.png")))
print("Selected model:", best_model_name)


In [ ]:
sweep = evaluate.threshold_sweep(best_pipeline, X_test, y_test)
plots.plot_threshold_sweep(sweep)
display(Image(str(config.FIGURES_DIR / "14_threshold_sweep.png")))
sweep.head(10)


The default 0.5 threshold is a convention, not a clinical decision. Lowering it trades
precision for recall — appropriate for a screening tool whose false positives get filtered
by a follow-up ultrasound, while false negatives walk out of the clinic undiagnosed.


## 8. The leakage experiment

This is the section that addresses the gap analysis directly.

Papers 4 and 5 report 98.9–99.3% accuracy on this same 541-row dataset. Our correctly
validated pipeline reaches noticeably less. Rather than hand-wave about why, we **measure**
the difference by running the same classifier under two protocols:

| Protocol | SMOTE + feature selection | What goes wrong |
|---|---|---|
| **Correct** | Inside the pipeline, refit per fold | Nothing — validation folds stay unseen |
| **Leaky** | Once, on the whole dataset, before CV | Synthetic minority rows interpolated from training patients land in the validation fold; the selector has already read every label |

Both use identical folds, identical data and an identical model. The only difference is
*where* the resampling happens.


In [ ]:
leakage = evaluate.leakage_experiment(X, y, rf)
plots.plot_leakage_experiment(leakage)
display(Image(str(config.FIGURES_DIR / "13_leakage_experiment.png")))
leakage.round(4)


**This is the headline finding of the project.** Simply moving SMOTE from inside the
cross-validation loop to outside it manufactures several points of accuracy and a much
larger jump in recall and F1 — without changing the model, the data, or the folds by
a single row.

That does not prove the reviewed papers made this mistake. Most of them do not describe
their protocol in enough detail to tell, which is precisely the problem. What it does show
is that a 98–99% headline on 541 records is *reachable* through methodology alone, so such
numbers cannot be taken at face value without a stated protocol.


## 9. Explainable AI with SHAP

Accuracy tells a clinician nothing actionable. SHAP decomposes each individual prediction
into signed per-feature contributions, answering both *what does the model rely on overall*
and *why this patient*.


In [ ]:
explain_name = "Random Forest"
explain_pipeline = test_models[explain_name]

shap_values, shap_data = explain.shap_values_for(explain_pipeline, X_test)
explain.plot_global_importance(shap_values, shap_data, explain_name)
explain.plot_beeswarm(shap_values, shap_data, explain_name)

display(Image(str(config.FIGURES_DIR / "16_shap_global_importance.png")))
display(Image(str(config.FIGURES_DIR / "17_shap_beeswarm.png")))


In [ ]:
importance = explain.importance_table(shap_values, shap_data)
importance.head(12).round(4)


In [ ]:
explain.plot_individual_explanations(explain_pipeline, X_test, y_test, shap_values, shap_data)
display(Image(str(config.FIGURES_DIR / "18_shap_waterfall_pcos_case.png")))
display(Image(str(config.FIGURES_DIR / "18_shap_waterfall_no_pcos_case.png")))


**Clinical sanity check.** The model's top features are follicle counts, then the
hyperandrogenism symptom cluster (skin darkening, hair growth), then cycle irregularity.
Those are, in order, the three legs of the Rotterdam criteria. The model independently
rediscovered the diagnostic standard from data rather than latching onto a spurious
correlate — which is a stronger argument for trusting it than any accuracy figure.

The `value_shap_corr` column gives direction: higher follicle counts push toward PCOS,
as expected. Note that mean *signed* SHAP would be misleading here — on a
majority-negative test set almost every feature has a negative mean, which reflects the
class balance rather than the feature.


## 9b. Is the hand-built pipeline leaving anything on the table?

The model comparison answers "which algorithm is best". It cannot answer the more useful
question: *is this whole approach underperforming?* Every entry shares the same pipeline,
so a common handicap would be invisible to the comparison.

[FLAML](https://github.com/microsoft/FLAML) (MIT licence) searches over algorithms *and*
hyperparameters under a time budget. The critical detail: it is given the **training split
only** and scored by the same protocol as everything else. An AutoML tool handed the full
dataset would tune itself against the test set — the same leak as Section 8, just
automated.


In [ ]:
automl_result = automl.run_automl_benchmark(X_train, y_train, X_test, y_test, time_budget=120)
our_row = test_results[test_results["model"] == best_model_name].iloc[0]

print("FLAML chose:", automl_result.get("best_estimator"))
print("its config :", automl_result.get("best_config"))

automl.comparison_table(automl_result, {
    "model": best_model_name,
    "accuracy": float(our_row["accuracy"]),
    "f1": float(our_row["f1"]),
    "roc_auc": float(our_row["roc_auc"]),
}).round(4)


An untargeted search did not beat a considered design — it landed slightly below, well
inside the +/-0.03 fold noise. FLAML independently converged on a Random Forest, mild
corroboration of the hand-built choice.

**The dataset is the binding constraint, not the pipeline.** Further hyperparameter tuning
on 541 rows is wasted effort, and any paper claiming a large gain from architecture search
on this data should be read sceptically.

*Caveat:* FLAML searches against a wall-clock budget, so the model it returns varies
between runs (we observed 0.935-0.943 test AUC). The conclusion is stable; the exact
number is not.


## 9c. Cost-tiered screening — what does each tier of testing buy?

The introduction to this project argues that PCOS testing "may not always be easy to
access, affordable, or quick, especially in areas with limited healthcare resources".
Nothing so far tests that claim. Here the features are grouped by what they **cost to
obtain**, and a model is trained on each cumulative tier:

* **questionnaire** — a form, a scale, a tape measure. No clinician, no equipment.
* **+ clinic vitals** — a nurse, five minutes.
* **+ blood panel** — venous draw and endocrine assays.
* **+ ultrasound** — a sonographer and a machine.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

def rf_plain():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                  random_state=config.RANDOM_STATE, n_jobs=-1)

tiers = clinical.tiered_models(X_train, y_train, X_test, y_test,
                               models.build_pipeline, rf_plain)
clinical.plot_tiered_models(tiers)
display(Image(str(config.FIGURES_DIR / "27_cost_tiers.png")))
tiers.round(4)


**This is the most clinically useful result in the project.**

1. A **questionnaire alone** reaches most of the full model's AUC — with no clinician,
   no laboratory and no ultrasound. That is a screening tool a health worker can run
   with a clipboard.
2. The **entire endocrine panel** adds almost nothing. FSH, LH, AMH, TSH, prolactin,
   vitamin D, progesterone, blood sugar and beta-HCG — nine assays and a venous draw —
   together buy about one hundredth of an AUC point for this task.
3. Only the **ultrasound** meaningfully pays for itself.

The honest deployment recommendation: run the questionnaire, skip the bloods, and spend
the budget on ultrasound access for the women the questionnaire flags. None of the five
reviewed papers ask this question, because they all optimise one model on all 41
features at once.


## 9d. Clinical utility, uncertainty, and subgroups

Three things accuracy cannot tell a clinician.


In [ ]:
# (a) Is using the model better than the two trivial policies?
curve = clinical.net_benefit(y_test, best_pipeline.predict_proba(X_test)[:, 1])
clinical.plot_decision_curve(curve, best_model_name)
display(Image(str(config.FIGURES_DIR / "26_decision_curve.png")))

useful = curve[curve["benefit_over_best_default"] > 0]
print(f"Model adds net benefit from {useful['threshold'].min():.0%} to {useful['threshold'].max():.0%}")
curve[curve["threshold"].isin([0.1, 0.2, 0.3, 0.5])].round(4)


**Net benefit** (Vickers & Elkin, 2006) puts the model on the same scale as the two
things a clinic could do instead — refer everyone, or refer nobody. The threshold
probability encodes how much worse a missed case is than a false alarm. A model is only
worth using where its curve sits above *both* defaults.


In [ ]:
# (b) How precise are the test-set numbers really?
ci = validation.bootstrap_test_metrics(best_pipeline, X_test, y_test, n_boot=2000)
validation.plot_bootstrap_ci(ci, best_model_name, len(y_test))
display(Image(str(config.FIGURES_DIR / "25_bootstrap_ci.png")))
ci


Recall — the metric that matters most for screening — is pinned down only to within
roughly ±12 points. That is the honest precision of *any* single-split number on a
dataset this size, including the 98-99% figures in the reviewed literature, none of
which report an interval.


In [ ]:
# (c) Does it work for everyone?
subgroups = clinical.subgroup_performance(best_pipeline, X_test, y_test)
clinical.plot_subgroups(subgroups)
display(Image(str(config.FIGURES_DIR / "28_subgroup_performance.png")))
subgroups


## 9e. Stability and the data ceiling

Two questions: would more data help, and how arbitrary is the "winner"?


In [ ]:
lc = validation.compute_learning_curve(
    models.build_pipeline(X_train, rf_plain(), selector=best_set, k_features=15),
    X_train, y_train,
)
validation.plot_learning_curve(lc, best_model_name)
display(Image(str(config.FIGURES_DIR / "24_learning_curve.png")))
lc.round(4)


In [ ]:
# Re-split the data 30 times and see which model wins each time.
sweep = validation.seed_sweep(X, y, models.build_all_models, n_seeds=30, selector=best_set)
wins = validation.win_counts(sweep)
validation.plot_seed_sweep(sweep, wins)
display(Image(str(config.FIGURES_DIR / "23_seed_sweep.png")))
wins.round(4)


**Every model wins sometimes.** The most frequent winner takes well under a third of the
draws, and each model's own test AUC swings by 0.08-0.15 depending only on which patients
landed in the test set.

Note especially where XGBoost sits here versus in the cross-validated table: a model can
finish last on average and still win the most individual splits. A paper reporting "model
X achieved the best accuracy" from a single 80/20 split is reporting a coin flip, and this
is what that coin flip looks like when you flip it thirty times.


## 9f. Nested cross-validation — applying our own standard to ourselves

Section 4 chose a feature set using cross-validation on the training split. Section 5
then reported a cross-validated score on that *same* split. Selection and evaluation
share data, so that number is optimistic in principle — the same class of error as the
SMOTE leak in Section 8, one level up.

Nested CV removes it: an inner loop picks the feature set, an outer loop scores the
result, and the two never share rows.


In [ ]:
nested = validation.nested_cv(X_train, y_train, models.build_pipeline, rf_plain)
validation.plot_nested_cv(nested)
display(Image(str(config.FIGURES_DIR / "22_nested_cv.png")))

print(f"Flat CV    : {nested['flat_mean']:.4f}  (selector chosen on the same data)")
print(f"Nested CV  : {nested['nested_mean']:.4f} +/- {nested['nested_std']:.4f}")
print(f"Optimism   : {nested['optimism']:+.4f}")


The bias turns out to be negligible, because the four feature sets perform almost
identically — choosing between them leaks very little. We flagged a real concern,
measured it, and found it immaterial *here*. Reporting the measurement rather than the
worry is the point: the same check on a study that tuned fifty hyperparameters would not
come out this way.


## 9g. External validation on an independent cohort

Every reviewed paper was criticised in our gap analysis for never testing on a second
population. This closes that gap as far as public data allows — and what it shows
reframes the criticism.

**The cohort:** a case-control study from Sfax, Tunisia (Mendeley,
doi:10.17632/tw34c7hv7z.1, CC BY 4.0). 88 women, same Rotterdam criteria, different
country, younger, heavier, near-balanced classes.

**The hazard:** many public "PCOS datasets" are re-uploads of the very Kerala file we
train on. Validating on a re-upload would look like external validation while being
nothing of the kind, so every candidate was provenance-checked first.


In [ ]:
print(external.SEARCH_LOG)


In [ ]:
ext = external.harmonise(external.load_external())
cohort_cmp = external.cohort_comparison(df, ext)
print(f"Shared features: {len(external.shared_feature_names())} of {X.shape[1]}")
print()
print("Deliberately excluded, with reasons:")
for name, reason in external.EXCLUDED_FEATURES.items():
    print(f"  {name}: {reason}")
cohort_cmp


In [ ]:
ext_results, ext_summary = external.validate_externally(df, ext, models.build_all_models)
external.plot_external_validation(ext_results, cohort_cmp)
display(Image(str(config.FIGURES_DIR / "21_external_validation.png")))
ext_results.round(4)


In [ ]:
print(external.EXTERNAL_DISCUSSION)


## 10. Honest comparison against the reviewed literature


In [ ]:
best_row = test_results[test_results["model"] == best_model_name].iloc[0]
cv_row = cv_results[cv_results["model"] == best_model_name].iloc[0]

comparison = literature.comparison_table(
    our_accuracy=float(best_row["accuracy"]),
    our_auc=float(best_row["roc_auc"]),
    our_model=best_model_name,
    our_accuracy_std=float(cv_row["accuracy_std"]),
)
plots.plot_paper_comparison(comparison)
display(Image(str(config.FIGURES_DIR / "15_paper_comparison.png")))

comparison[["paper", "best_model", "reported_accuracy", "validation_protocol",
            "imbalance_handled", "explainability"]]


In [ ]:
print(literature.DISCUSSION)


## 11. Conclusions and limitations

**What was delivered against the proposal**

| Objective | Status |
|---|---|
| Preprocessing + EDA pipeline | Done — 5 classes of data defect fixed, 7 EDA figures |
| Compare correlation / Chi-Square / RFE | Done — all three, cross-validated, plus a baseline |
| Address class imbalance explicitly | Done — SMOTE inside folds, with a 3-way ablation |
| Multiple models + ensemble, stratified k-fold | Done — 5 models, 10-fold × 3 repeats |
| SHAP explainability | Done — global, directional and per-patient |
| Honest comparison with the five papers | Done — plus a measured leakage experiment |

**Delivered beyond the proposal**

| Addition | Why it matters |
|---|---|
| Nine algorithms, not five | Spans linear / kernel / probabilistic / instance-based / bagged / boosted / stacked, so "they're tied" is a real finding rather than five variants of one idea |
| Paired significance tests + 30-split sweep | Turns "the error bars overlap" into a measured claim; every model wins some splits |
| Calibration analysis | The one axis on which the tied models genuinely differ — and which no reviewed paper reports |
| AutoML benchmark | Tests the pipeline itself for headroom, not just the algorithm choice |
| CatBoost included | Paper 3's best model, so its 95.7% is tested under our protocol rather than quoted |
| **Cost-tiered screening** | Quantifies what ultrasound and bloodwork actually buy — the deployable result |
| **External validation** | An independent Tunisian cohort, with provenance checks against re-uploads |
| Nested CV | Applies our own anti-leakage standard to our own model-selection step |
| Bootstrap CIs + decision curves + subgroups | Uncertainty, clinical utility and fairness — none reported by any reviewed paper |
| Physiological range checks | Caught 13 impossible values (BP of 12/80, pulse of 13 bpm, FSH of 5052) that the four papers using this dataset presumably trained on |
| TRIPOD+AI checklist | The clinical prediction reporting standard, filled in with a risk-of-bias self-assessment |

**Limitations, stated plainly**

1. **External validation is partial, and structurally limited.** We did validate on an
   independent Tunisian cohort — but only 8 features are shared, so what transferred was
   a weak restricted model, not the headline one. The headline model cannot be externally
   validated on any currently public data, because no public cohort records follicle
   counts and PCOS symptoms in a compatible schema.

1b. **Circularity between predictor and label.** Follicle count is our strongest SHAP
   feature *and* one of the three Rotterdam criteria used to assign the label. The full
   model is best read as automating consistent application of the diagnostic criteria,
   not as discovering new biology. The questionnaire-only model is the more scientifically
   interesting one, because none of its inputs are diagnostic criteria.
2. **Small sample.** 541 rows, 109 in the test set, 36 of them PCOS cases. Three patients
   changing sides moves test accuracy by ~3 points. Every test-set number here should be
   read with that granularity in mind — the cross-validated figures are more trustworthy.
3. **No ultrasound imagery.** Also identified in the gap analysis and also not addressed;
   the dataset contains follicle *counts* derived from ultrasound, but not the images.
4. **Self-reported symptoms.** Several of the strongest features (weight gain, hair
   growth, fast food) are patient-reported and may be recorded *after* a clinical
   suspicion of PCOS has formed, which would make them partly a consequence of the
   diagnosis rather than a predictor of it.
5. **Single-centre labels.** All diagnoses come from one region's clinical practice.

**The main takeaway**

A correctly validated model on this dataset lands around 0.95 ROC-AUC and ~90% accuracy —
strong, clinically plausible, and consistent with the Rotterdam criteria under SHAP. The
98–99% figures in the literature are reachable, but our leakage experiment shows a large
share of that margin can be produced by protocol choices alone. Reporting the lower,
defensible number is the more useful scientific contribution.

Three findings support that, and each is a measurement rather than an assertion:

1. **A leaky protocol manufactures +3.2 points of accuracy and +9.0 F1** with no change to
   model, data or folds.
2. **Zero of eight algorithms differ practically from the best.** Chasing the leaderboard
   on this dataset is chasing noise — including the stacking ensembles that two of the
   reviewed papers crown.
3. **A 120-second AutoML search found no headroom.** The dataset, not the pipeline, is
   the ceiling.

Taken together: the interesting variable in PCOS prediction on this dataset is not the
algorithm. It is the protocol.
